# NumPy — triki i mechanika, ktore pojawiaja sie w quant finance

Ten notebook nie jest o finansach — jest o samym NumPy. Skupia sie na operacjach,
ktore wygladaja magicznie przy pierwszym kontakcie, ale maja prosta logike pod spodem.

Kazda sekcja: co to robi -> przyklad minimalny -> przyklad jak w naszych notebookach quant.

Tematy:
1. Slicing z krokiem: [::-1] i [::k]
2. np.newaxis i broadcasting
3. Generowanie probek z rozkladow
4. Mnozenie macierzy: @, np.dot, kiedy ktore
5. Indeksowanie tablica indeksow: arr[idx] i argsort
6. np.outer, np.diag, mieszanie wymiarow
7. Redukcje z axis: sum, mean, std po wierszach/kolumnach
8. Maski logiczne i np.where

---


## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__)


NumPy 2.4.4


---
## 1. Slicing z krokiem: arr[start:stop:step]

Pelna skladnia slice to arr[start:stop:step]. Wszystkie trzy parametry sa opcjonalne:
brak start -> od pierwszego elementu, brak stop -> do ostatniego, brak step -> krok 1.

step moze byc ujemny -- wtedy idziesz od konca do przodu.


In [2]:
arr = np.array([10, 20, 30, 40, 50])

print("arr            ", arr)
print("arr[::1]       ", arr[::1])
print("arr[::2]       ", arr[::2])
print("arr[::-1]      ", arr[::-1])
print("arr[1:4]       ", arr[1:4])
print("arr[1:4:2]     ", arr[1:4:2])
print("arr[::-2]      ", arr[::-2])


arr             [10 20 30 40 50]
arr[::1]        [10 20 30 40 50]
arr[::2]        [10 30 50]
arr[::-1]       [50 40 30 20 10]
arr[1:4]        [20 30 40]
arr[1:4:2]      [20 40]
arr[::-2]       [50 30 10]


### Dlaczego [::-1] odwraca cala tablice

arr[::-1] zaczyna od ostatniego indeksu i idzie krokiem -1 do samego poczatku.
To jest tani trik -- nie kopiuje danych, tylko tworzy widok (view) z innym krokiem.


In [3]:
original = np.array([1, 2, 3, 4, 5])
reversed_view = original[::-1]

print("original:       ", original)
print("reversed_view:  ", reversed_view)

reversed_view[0] = 999
print("\nPo modyfikacji reversed_view[0] = 999:")
print("original:       ", original)
print("reversed_view:  ", reversed_view)
print(f"\nCzy to ten sam bufor pamieci? {np.shares_memory(original, reversed_view)}")


original:        [1 2 3 4 5]
reversed_view:   [5 4 3 2 1]

Po modyfikacji reversed_view[0] = 999:
original:        [  1   2   3   4 999]
reversed_view:   [999   4   3   2   1]

Czy to ten sam bufor pamieci? True


### Przyklad jak w naszych notebookach: np.argsort(...)[::-1]

To byl jeden z trikow przy PCA -- sortowanie eigenvalues malejaco.
np.linalg.eigh zwraca eigenvalues rosnaco, a w PCA chcemy malejaco.


In [4]:
evals = np.array([0.02, 0.50, 0.01, 0.30])

idx_ascending = np.argsort(evals)
print("argsort (rosnaco):  ", idx_ascending)
print("evals[idx_ascending]:", evals[idx_ascending])

idx_descending = idx_ascending[::-1]
print("\nidx po [::-1]:      ", idx_descending)
print("evals[idx_descending]:", evals[idx_descending])

idx_one_line = np.argsort(evals)[::-1]
print("\nJedna linia daje to samo:", np.array_equal(idx_one_line, idx_descending))


argsort (rosnaco):   [2 0 3 1]
evals[idx_ascending]: [0.01 0.02 0.3  0.5 ]

idx po [::-1]:       [1 3 0 2]
evals[idx_descending]: [0.5  0.3  0.02 0.01]

Jedna linia daje to samo: True


In [5]:
evecs = np.array([
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8],
])

idx = np.argsort(evals)[::-1]
evals_sorted = evals[idx]
evecs_sorted = evecs[:, idx]

print("evals oryginalne:", evals)
print("evals sorted:    ", evals_sorted)
print()
print("evecs oryginalne:\n", evecs)
print("evecs_sorted (kolumny przestawione):\n", evecs_sorted)
print()
print("evals_sorted[0] =", evals_sorted[0], " -> evecs_sorted[:,0] =", evecs_sorted[:, 0])


evals oryginalne: [0.02 0.5  0.01 0.3 ]
evals sorted:     [0.5  0.3  0.02 0.01]

evecs oryginalne:
 [[0.1 0.2 0.3 0.4]
 [0.5 0.6 0.7 0.8]]
evecs_sorted (kolumny przestawione):
 [[0.2 0.4 0.1 0.3]
 [0.6 0.8 0.5 0.7]]

evals_sorted[0] = 0.5  -> evecs_sorted[:,0] = [0.2 0.6]


---
## 2. np.newaxis i broadcasting

NumPy potrafi wykonywac operacje na tablicach roznych kształtow, jesli sa kompatybilne.
Zasada (od ostatniego wymiaru): wymiary sa kompatybilne jesli sa rowne albo jeden z nich
to 1. Wymiar=1 jest rozciagany do rozmiaru drugiej tablicy. Brakujace wymiary (z lewej)
sa dodawane jako 1.

np.newaxis to po prostu None uzyty do dodania nowego wymiaru rozmiaru 1.


In [6]:
v = np.array([1, 2, 3])
print("v.shape:                ", v.shape)
print("v[:, np.newaxis].shape: ", v[:, np.newaxis].shape)
print("v[np.newaxis, :].shape: ", v[np.newaxis, :].shape)
print()
print("v:                  ", v)
print("v[:, np.newaxis]:\n", v[:, np.newaxis])
print("v[np.newaxis, :]:   ", v[np.newaxis, :])


v.shape:                 (3,)
v[:, np.newaxis].shape:  (3, 1)
v[np.newaxis, :].shape:  (1, 3)

v:                   [1 2 3]
v[:, np.newaxis]:
 [[1]
 [2]
 [3]]
v[np.newaxis, :]:    [[1 2 3]]


In [7]:
a = np.array([1, 2, 3])
b = np.array([10, 20])

a_col = a[:, np.newaxis]
b_row = b[np.newaxis, :]

print("a_col.shape:", a_col.shape, " b_row.shape:", b_row.shape)

result = a_col * b_row
print("\nWynik (3,2):")
print(result)
print()
print("Reczna weryfikacja: result[i,j] = a[i] * b[j]")
for i in range(3):
    for j in range(2):
        print(f"  result[{i},{j}] = a[{i}]*b[{j}] = {a[i]}*{b[j]} = {result[i,j]}")


a_col.shape: (3, 1)  b_row.shape: (1, 2)

Wynik (3,2):
[[10 20]
 [20 40]
 [30 60]]

Reczna weryfikacja: result[i,j] = a[i] * b[j]
  result[0,0] = a[0]*b[0] = 1*10 = 10
  result[0,1] = a[0]*b[1] = 1*20 = 20
  result[1,0] = a[1]*b[0] = 2*10 = 20
  result[1,1] = a[1]*b[1] = 2*20 = 40
  result[2,0] = a[2]*b[0] = 3*10 = 30
  result[2,1] = a[2]*b[1] = 3*20 = 60


### Reguly broadcastingu w punktach

Porownujemy shapes od prawej:
(3,1) i (1,2) -> wynik (3,2): kazdy wymiar to max(3,1)=3, max(1,2)=2.
Jezeli wymiary nie sa rowne ani jeden z nich nie jest 1 -> blad.


In [8]:
def test_broadcast(shape1, shape2):
    try:
        x = np.zeros(shape1)
        y = np.zeros(shape2)
        z = x + y
        print(f"{shape1} + {shape2}  ->  OK, wynik {z.shape}")
    except ValueError as e:
        print(f"{shape1} + {shape2}  ->  BLAD: {e}")

test_broadcast((3, 1), (1, 4))
test_broadcast((3, 4), (4,))
test_broadcast((3, 4), (3,))
test_broadcast((3, 4), (3, 1))
test_broadcast((5, 3), (4, 3))


(3, 1) + (1, 4)  ->  OK, wynik (3, 4)
(3, 4) + (4,)  ->  OK, wynik (3, 4)
(3, 4) + (3,)  ->  BLAD: operands could not be broadcast together with shapes (3,4) (3,) 
(3, 4) + (3, 1)  ->  OK, wynik (3, 4)
(5, 3) + (4, 3)  ->  BLAD: operands could not be broadcast together with shapes (5,3) (4,3) 


In [9]:
betas         = np.array([0.8, 1.2, 1.0])
market_factor = np.array([0.01, -0.02, 0.03, 0.00])

try:
    bad = betas * market_factor
except ValueError as e:
    print(f"betas * market_factor (bez newaxis): BLAD -- {e}")

result = betas[:, np.newaxis] * market_factor[np.newaxis, :]
print(f"\nbetas[:, newaxis].shape:         {betas[:, np.newaxis].shape}")
print(f"market_factor[newaxis, :].shape: {market_factor[np.newaxis, :].shape}")
print(f"Wynik shape: {result.shape}")
print(result)


betas * market_factor (bez newaxis): BLAD -- operands could not be broadcast together with shapes (3,) (4,) 

betas[:, newaxis].shape:         (3, 1)
market_factor[newaxis, :].shape: (1, 4)
Wynik shape: (3, 4)
[[ 0.008 -0.016  0.024  0.   ]
 [ 0.012 -0.024  0.036  0.   ]
 [ 0.01  -0.02   0.03   0.   ]]


In [10]:
mu = np.array([0.001, 0.002, 0.003])
L_Z = np.array([
    [0.01, -0.02, 0.03, 0.00, 0.01],
    [0.02,  0.01, -0.01, 0.02, 0.00],
    [-0.01, 0.00, 0.02, -0.01, 0.01],
])

scenarios = mu[:, np.newaxis] + L_Z
print("mu[:, np.newaxis].shape:", mu[:, np.newaxis].shape)
print("L_Z.shape:               ", L_Z.shape)
print("scenarios.shape:         ", scenarios.shape)
print()
print(scenarios)
print()
print("Weryfikacja: kolumna 0 to mu + L_Z[:,0]")
print(mu + L_Z[:, 0])


mu[:, np.newaxis].shape: (3, 1)
L_Z.shape:                (3, 5)
scenarios.shape:          (3, 5)

[[ 0.011 -0.019  0.031  0.001  0.011]
 [ 0.022  0.012 -0.008  0.022  0.002]
 [-0.007  0.003  0.023 -0.007  0.013]]

Weryfikacja: kolumna 0 to mu + L_Z[:,0]
[ 0.011  0.022 -0.007]


---
## 3. Generowanie probek z rozkladow

np.random.normal(loc, scale, size) generuje probki z rozkladu normalnego N(loc, scale^2).
loc = srednia (mu), scale = odchylenie standardowe (sigma) -- NIE wariancja.
size = ksztalt wynikowej tablicy.


In [11]:
single_value = np.random.normal(0, 1)
small_array  = np.random.normal(0, 1, 5)
matrix       = np.random.normal(0, 1, (3, 4))

print("Jedna wartosc:", single_value)
print("Wektor (5,):  ", small_array)
print("Macierz (3,4):\n", matrix)


Jedna wartosc: 0.4967141530112327
Wektor (5,):   [-0.1383  0.6477  1.523  -0.2342 -0.2341]
Macierz (3,4):
 [[ 1.5792  0.7674 -0.4695  0.5426]
 [-0.4634 -0.4657  0.242  -1.9133]
 [-1.7249 -0.5623 -1.0128  0.3142]]


### Trik: loc i scale moga byc tablicami -- rozne parametry dla kazdego wymiaru

To bylo w notebooku: rozne srednie/vol dla 3 aktywow naraz.


In [12]:
means = [0.0004, 0.0006, 0.0003]
stds  = [0.015,  0.020,  0.012]

returns = np.random.normal(loc=means, scale=stds, size=(252, 3))
print(f"Shape: {returns.shape}")
print(f"Kolumna 0 (powinna miec mean~{means[0]}, std~{stds[0]}):")
print(f"  empiryczna srednia: {returns[:,0].mean():.5f}")
print(f"  empiryczne std:     {returns[:,0].std():.5f}")
print()
print(f"Kolumna 1 (powinna miec mean~{means[1]}, std~{stds[1]}):")
print(f"  empiryczna srednia: {returns[:,1].mean():.5f}")
print(f"  empiryczne std:     {returns[:,1].std():.5f}")


Shape: (252, 3)
Kolumna 0 (powinna miec mean~0.0004, std~0.015):
  empiryczna srednia: 0.00117
  empiryczne std:     0.01352

Kolumna 1 (powinna miec mean~0.0006, std~0.02):
  empiryczna srednia: -0.00198
  empiryczne std:     0.02002


### np.random.standard_normal(size) vs np.random.normal(0, 1, size)

To samo, ale standard_normal jest nieco szybsze. Generuje zawsze N(0,1).


In [13]:
import time

N = 10_000_000

t0 = time.perf_counter()
_ = np.random.standard_normal(N)
t_standard = time.perf_counter() - t0

t0 = time.perf_counter()
_ = np.random.normal(0, 1, N)
t_normal = time.perf_counter() - t0

print(f"standard_normal: {t_standard*1000:.2f} ms")
print(f"normal(0,1,...): {t_normal*1000:.2f} ms")


standard_normal: 590.20 ms
normal(0,1,...): 748.46 ms


### np.random.multivariate_normal(mean, cov, size)

Generuje probki z wielowymiarowego rozkladu normalnego naraz -- z wbudowana korelacja
zadana przez macierz kowariancji.


In [14]:
mean_2d = [0, 0]
cov_2d  = [[1.0, 0.7],
           [0.7, 1.0]]

samples = np.random.multivariate_normal(mean_2d, cov_2d, size=1000)
print(f"Shape: {samples.shape}")

empirical_corr = np.corrcoef(samples.T)
print(f"\nKorelacja empiryczna (cel: 0.7):\n{empirical_corr}")


Shape: (1000, 2)

Korelacja empiryczna (cel: 0.7):
[[1.     0.7036]
 [0.7036 1.    ]]


In [ ]:
L = np.linalg.cholesky(cov_2d)
Z = np.random.standard_normal((2, 1000))
X = (L @ Z).T + np.array(mean_2d)

print("Korelacja z reczne Cholesky:")
print(np.corrcoef(X.T))
print()
print("multivariate_normal i reczny Cholesky daja TEN SAM mechanizm")
print("(rozne liczby bo inne ziarno losowosci, ale ta sama struktura statystyczna)")


---
## 4. Mnozenie macierzy: @, np.dot, *

| Operator | Nazwa | Co robi |
|----------|-------|---------|
| * | element-wise | mnozy odpowiadajace sobie elementy (broadcasting) |
| @ lub np.matmul | mnozenie macierzowe | standardowe mnozenie z algebry liniowej |
| np.dot | dot product / matmul | dla 1D: dot product; dla 2D: to samo co @ |


In [16]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("A * B  (element-wise):")
print(A * B)
print()
print("A @ B  (mnozenie macierzowe):")
print(A @ B)
print()
print("Reczna weryfikacja A@B[0,0] = A[0,0]*B[0,0] + A[0,1]*B[1,0]:")
print(f"  {A[0,0]}*{B[0,0]} + {A[0,1]}*{B[1,0]} = {A[0,0]*B[0,0] + A[0,1]*B[1,0]}")


A * B  (element-wise):
[[ 5 12]
 [21 32]]

A @ B  (mnozenie macierzowe):
[[19 22]
 [43 50]]

Reczna weryfikacja A@B[0,0] = A[0,0]*B[0,0] + A[0,1]*B[1,0]:
  1*5 + 2*7 = 19


In [17]:
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])

print("v1 @ v2:      ", v1 @ v2)
print("np.dot(v1,v2):", np.dot(v1, v2))
print("Recznie:      ", (v1*v2).sum())

print(f"\nTyp wyniku v1@v2: {type(v1 @ v2)}")


v1 @ v2:       32
np.dot(v1,v2): 32
Recznie:       32

Typ wyniku v1@v2: <class 'numpy.int64'>


In [18]:
w        = np.array([0.3, 0.5, 0.2])
returns  = np.random.normal(0, 0.01, (252, 3))

port_returns = returns @ w
print(f"returns @ w:        shape {port_returns.shape}  (252 wyniki, jeden na dzien)")

Sigma = np.cov(returns.T)

var_portfolio = w @ Sigma @ w
print(f"w @ Sigma @ w:      {var_portfolio:.8f}  (skalar)")

L = np.linalg.cholesky(Sigma)
Z = np.random.standard_normal((3, 1000))
X = L @ Z
print(f"L @ Z:              shape {X.shape}")


returns @ w:        shape (252,)  (252 wyniki, jeden na dzien)
w @ Sigma @ w:      0.00003520  (skalar)
L @ Z:              shape (3, 1000)


### Trik: A * v (broadcasting) vs A @ v (mnozenie macierzowe) -- LATWO POMYLIC


In [19]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
v = np.array([10, 100, 1000])

elementwise = A * v
print("A * v  (broadcasting, mnozy kolumny):")
print(elementwise)
print("Shape:", elementwise.shape, " <- (2,3), TEN SAM shape co A")

print()

matmul = A @ v
print("A @ v  (mnozenie macierzowe):")
print(matmul)
print("Shape:", matmul.shape, " <- (2,), redukcja wymiaru!")

print()
print("To sa DWIE ROZNE operacje matematyczne, mimo podobnego zapisu w kodzie.")


A * v  (broadcasting, mnozy kolumny):
[[  10  200 3000]
 [  40  500 6000]]
Shape: (2, 3)  <- (2,3), TEN SAM shape co A

A @ v  (mnozenie macierzowe):
[3210 6540]
Shape: (2,)  <- (2,), redukcja wymiaru!

To sa DWIE ROZNE operacje matematyczne, mimo podobnego zapisu w kodzie.


---
## 5. Indeksowanie tablica indeksow (fancy indexing)

arr[idx] gdzie idx jest tablica liczb calkowitych -- zwraca nowa tablice, w ktorej
elementy sa przestawione/wybrane zgodnie z idx.


In [20]:
arr = np.array([100, 200, 300, 400, 500])
idx = np.array([3, 0, 0, 4])

print("arr:     ", arr)
print("idx:     ", idx)
print("arr[idx]:", arr[idx])
print()
print("Logika: arr[idx][k] = arr[idx[k]]")
print("  arr[idx][0] = arr[3] =", arr[3])
print("  arr[idx][1] = arr[0] =", arr[0])
print("  arr[idx][2] = arr[0] =", arr[0], "  <- powtorzenia sa OK")
print("  arr[idx][3] = arr[4] =", arr[4])


arr:      [100 200 300 400 500]
idx:      [3 0 0 4]
arr[idx]: [400 100 100 500]

Logika: arr[idx][k] = arr[idx[k]]
  arr[idx][0] = arr[3] = 400
  arr[idx][1] = arr[0] = 100
  arr[idx][2] = arr[0] = 100   <- powtorzenia sa OK
  arr[idx][3] = arr[4] = 500


In [21]:
matrix = np.array([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
])

idx = np.array([2, 0, 1])

print("matrix:\n", matrix)
print()
print("matrix[idx]  -- permutuje WIERSZE:")
print(matrix[idx])
print()
print("matrix[:, idx]  -- permutuje KOLUMNY:")
print(matrix[:, idx])


matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

matrix[idx]  -- permutuje WIERSZE:
[[ 9 10 11 12]
 [ 1  2  3  4]
 [ 5  6  7  8]]

matrix[:, idx]  -- permutuje KOLUMNY:
[[ 3  1  2]
 [ 7  5  6]
 [11  9 10]]


In [22]:
values = np.array([30, 10, 50, 20, 40])

sort_idx = np.argsort(values)
print("values:    ", values)
print("argsort:   ", sort_idx)
print("values[idx]:", values[sort_idx])

print()
print("Roznica vs np.sort (ktore sortuje WARTOSCI bezposrednio):")
print("np.sort(values):", np.sort(values))
print("Identyczne:", np.array_equal(np.sort(values), values[sort_idx]))


values:     [30 10 50 20 40]
argsort:    [1 3 0 4 2]
values[idx]: [10 20 30 40 50]

Roznica vs np.sort (ktore sortuje WARTOSCI bezposrednio):
np.sort(values): [10 20 30 40 50]
Identyczne: True


In [23]:
evals = np.array([0.05, 0.40, 0.02, 0.30, 0.01])
evecs = np.random.normal(0, 1, (5, 5))

print("evals oryginalne:", evals)

idx = np.argsort(evals)
print("argsort (rosnaco):", idx)

idx_desc = idx[::-1]
print("po [::-1] (malejaco):", idx_desc)

evals_sorted = evals[idx_desc]
evecs_sorted = evecs[:, idx_desc]

print()
print("evals_sorted:", evals_sorted)
print("Sprawdzenie: evals_sorted[0] to najwieksza wartosc:", evals_sorted[0] == evals.max())
print("evecs_sorted.shape:", evecs_sorted.shape, " (taki sam jak oryginal, tylko kolumny przestawione)")


evals oryginalne: [0.05 0.4  0.02 0.3  0.01]
argsort (rosnaco): [4 2 0 3 1]
po [::-1] (malejaco): [1 3 0 2 4]

evals_sorted: [0.4  0.3  0.05 0.02 0.01]
Sprawdzenie: evals_sorted[0] to najwieksza wartosc: True
evecs_sorted.shape: (5, 5)  (taki sam jak oryginal, tylko kolumny przestawione)


---
## 6. np.outer, np.diag -- budowanie macierzy z wektorow

np.outer(a, b) to iloczyn zewnetrzny: outer[i,j] = a[i] * b[j]. Wynik to macierz
(len(a), len(b)).


In [24]:
a = np.array([1, 2, 3])
b = np.array([10, 20])

result = np.outer(a, b)
print("np.outer(a, b):")
print(result)
print()
print("Recznie: outer[i,j] = a[i]*b[j]")
for i in range(3):
    for j in range(2):
        print(f"  outer[{i},{j}] = {a[i]}*{b[j]} = {result[i,j]}")

print()
print("To samo co a[:, np.newaxis] * b[np.newaxis, :]:")
print(a[:, np.newaxis] * b[np.newaxis, :])


np.outer(a, b):
[[10 20]
 [20 40]
 [30 60]]

Recznie: outer[i,j] = a[i]*b[j]
  outer[0,0] = 1*10 = 10
  outer[0,1] = 1*20 = 20
  outer[1,0] = 2*10 = 20
  outer[1,1] = 2*20 = 40
  outer[2,0] = 3*10 = 30
  outer[2,1] = 3*20 = 60

To samo co a[:, np.newaxis] * b[np.newaxis, :]:
[[10 20]
 [20 40]
 [30 60]]


In [25]:
corr = np.array([
    [1.0, 0.5, 0.3],
    [0.5, 1.0, 0.4],
    [0.3, 0.4, 1.0]
])
stds = np.array([0.15, 0.20, 0.10])

std_outer = np.outer(stds, stds)
print("np.outer(stds, stds):")
print(std_outer)
print()

Sigma_from_corr = corr * std_outer
print("Sigma (z korelacji i std):")
print(Sigma_from_corr)


np.outer(stds, stds):
[[0.0225 0.03   0.015 ]
 [0.03   0.04   0.02  ]
 [0.015  0.02   0.01  ]]

Sigma (z korelacji i std):
[[0.0225 0.015  0.0045]
 [0.015  0.04   0.008 ]
 [0.0045 0.008  0.01  ]]


### np.diag -- dwie role naraz (przeciazona funkcja)

Jesli wejscie to wektor (1D) -> tworzy macierz diagonalna z tym wektorem na przekatnej.
Jesli wejscie to macierz (2D) -> wyciaga przekatna jako wektor.


In [26]:
v = np.array([1, 2, 3])

diag_matrix = np.diag(v)
print("np.diag(wektor) -> macierz diagonalna:")
print(diag_matrix)

print()

M = np.array([[10, 1, 2],
              [3, 20, 4],
              [5, 6, 30]])
diag_vector = np.diag(M)
print("np.diag(macierz) -> wektor (przekatna):")
print(diag_vector)


np.diag(wektor) -> macierz diagonalna:
[[1 0 0]
 [0 2 0]
 [0 0 3]]

np.diag(macierz) -> wektor (przekatna):
[10 20 30]


In [27]:
Sigma_projected = np.array([
    [0.04, 0.001, -0.002],
    [0.001, 0.09, 0.0005],
    [-0.002, 0.0005, 0.0625]
])

diagonal_only = np.diag(np.diag(Sigma_projected))

print("np.diag(Sigma_projected) [wektor]:", np.diag(Sigma_projected))
print()
print("np.diag(np.diag(Sigma_projected)) [macierz z zerami poza przekatna]:")
print(diagonal_only)
print()

off_diagonal = Sigma_projected - diagonal_only
print("Sigma_projected - to:  (zostaje TYLKO to co bylo poza przekatna)")
print(off_diagonal)


np.diag(Sigma_projected) [wektor]: [0.04   0.09   0.0625]

np.diag(np.diag(Sigma_projected)) [macierz z zerami poza przekatna]:
[[0.04   0.     0.    ]
 [0.     0.09   0.    ]
 [0.     0.     0.0625]]

Sigma_projected - to:  (zostaje TYLKO to co bylo poza przekatna)
[[ 0.      0.001  -0.002 ]
 [ 0.001   0.      0.0005]
 [-0.002   0.0005  0.    ]]


---
## 7. Redukcje z axis: sum/mean/std po wierszach lub kolumnach

axis okresla wzdluz ktorego wymiaru funkcja zwija dane. Mysl o tym jako:
"ktory wymiar znika po tej operacji?"


In [28]:
M = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print("M:\n", M)
print()
print("M.sum()        -- caly array, skalar:    ", M.sum())
print("M.sum(axis=0)  -- redukcja wierszy (0):  ", M.sum(axis=0))
print("M.sum(axis=1)  -- redukcja kolumn (1):   ", M.sum(axis=1))
print()
print("axis=0: 'spada' pierwszy wymiar (wiersze), zostaja kolumny -> wynik shape (3,)")
print("axis=1: 'spada' drugi wymiar (kolumny), zostaja wiersze   -> wynik shape (2,)")


M:
 [[1 2 3]
 [4 5 6]]

M.sum()        -- caly array, skalar:     21
M.sum(axis=0)  -- redukcja wierszy (0):   [5 7 9]
M.sum(axis=1)  -- redukcja kolumn (1):    [ 6 15]

axis=0: 'spada' pierwszy wymiar (wiersze), zostaja kolumny -> wynik shape (3,)
axis=1: 'spada' drugi wymiar (kolumny), zostaja wiersze   -> wynik shape (2,)


In [29]:
returns_matrix = np.random.normal(0.0005, 0.015, (252, 3))

mean_per_asset = returns_matrix.mean(axis=0)
print("mean(axis=0) -- srednia kazdego aktywa:", mean_per_asset.round(5))
print("shape:", mean_per_asset.shape, " <- 3 liczby, jedna per aktywo")

print()

mean_per_day = returns_matrix.mean(axis=1)
print("mean(axis=1) -- srednia po aktywach kazdego dnia, pierwsze 5:", mean_per_day[:5].round(5))
print("shape:", mean_per_day.shape, " <- 252 liczby, jedna per dzien")


mean(axis=0) -- srednia kazdego aktywa: [ 0.0008  0.001  -0.0001]
shape: (3,)  <- 3 liczby, jedna per aktywo

mean(axis=1) -- srednia po aktywach kazdego dnia, pierwsze 5: [-0.0028  0.0008 -0.0006  0.0011  0.0035]
shape: (252,)  <- 252 liczby, jedna per dzien


In [30]:
print("BEZ axis (caly array, jedna liczba):", returns_matrix.mean())
print("Z axis=0 (per aktywo, 3 liczby):    ", returns_matrix.mean(axis=0).round(5))
print()
print("Gdybysmy chcieli 'roczna vol kazdego aktywa', MUSIMY uzyc axis=0:")
ann_vol_per_asset = returns_matrix.std(axis=0) * np.sqrt(252)
print(ann_vol_per_asset.round(4))


BEZ axis (caly array, jedna liczba): 0.0005822257786653035
Z axis=0 (per aktywo, 3 liczby):     [ 0.0008  0.001  -0.0001]

Gdybysmy chcieli 'roczna vol kazdego aktywa', MUSIMY uzyc axis=0:
[0.2417 0.2146 0.234 ]


### keepdims=True -- zachowanie wymiaru dla poprawnego broadcastingu

Bylo uzyte przy centrowaniu danych: X - X.mean(axis=1, keepdims=True)


In [31]:
X = np.array([
    [1.0, 2.0, 3.0],
    [10.0, 20.0, 30.0]
])

mean_no_keepdims = X.mean(axis=1)
mean_with_keepdims = X.mean(axis=1, keepdims=True)

print("mean(axis=1).shape:                ", mean_no_keepdims.shape)
print("mean(axis=1, keepdims=True).shape: ", mean_with_keepdims.shape)
print()

X_centered = X - mean_with_keepdims
print("X - mean(keepdims=True):")
print(X_centered)

try:
    bad = X - mean_no_keepdims
except ValueError as e:
    print(f"\nX - mean(axis=1) bez keepdims: BLAD -- {e}")


mean(axis=1).shape:                 (2,)
mean(axis=1, keepdims=True).shape:  (2, 1)

X - mean(keepdims=True):
[[ -1.   0.   1.]
 [-10.   0.  10.]]

X - mean(axis=1) bez keepdims: BLAD -- operands could not be broadcast together with shapes (2,3) (2,) 


---
## 8. Maski logiczne i np.where

Porownanie tablicy z wartoscia daje tablice boolowska tego samego shape. Mozna jej
uzyc do filtrowania (boolean indexing) albo do liczenia (bo True=1, False=0).


In [32]:
returns = np.array([0.02, -0.01, 0.03, -0.04, 0.01, -0.02])

mask_negative = returns < 0
print("returns:      ", returns)
print("mask (< 0):   ", mask_negative)
print()

negative_returns = returns[mask_negative]
print("returns[mask]:", negative_returns)

count_negative = mask_negative.sum()
print(f"\nLiczba ujemnych dni: {count_negative}")
print(f"Procent ujemnych dni: {mask_negative.mean()*100:.1f}%")


returns:       [ 0.02 -0.01  0.03 -0.04  0.01 -0.02]
mask (< 0):    [False  True False  True False  True]

returns[mask]: [-0.01 -0.04 -0.02]

Liczba ujemnych dni: 3
Procent ujemnych dni: 50.0%


In [33]:
payoff_call = np.where(returns > 0, returns, 0)
print("returns:    ", returns)
print("np.where(returns>0, returns, 0):", payoff_call)

print("\nTo samo przez np.maximum(returns, 0):", np.maximum(returns, 0))


returns:     [ 0.02 -0.01  0.03 -0.04  0.01 -0.02]
np.where(returns>0, returns, 0): [0.02 0.   0.03 0.   0.01 0.  ]

To samo przez np.maximum(returns, 0): [0.02 0.   0.03 0.   0.01 0.  ]


In [34]:
N = 4
corr_matrix = np.array([
    [1.00, 0.65, 0.50, 0.30],
    [0.65, 1.00, 0.70, 0.40],
    [0.50, 0.70, 1.00, 0.60],
    [0.30, 0.40, 0.60, 1.00],
])

identity_mask = np.eye(N, dtype=bool)
off_diagonal_mask = ~identity_mask

print("np.eye(4, dtype=bool) -- przekatna:")
print(identity_mask)
print()
print("~identity_mask -- wszystko OPROCZ przekatnej:")
print(off_diagonal_mask)
print()

mean_corr_excl_diag = corr_matrix[off_diagonal_mask].mean()
mean_corr_incl_diag = corr_matrix.mean()

print(f"Srednia korelacja (z przekatna, ZLE):    {mean_corr_incl_diag:.4f}")
print(f"Srednia korelacja (bez przekatnej, OK): {mean_corr_excl_diag:.4f}")


np.eye(4, dtype=bool) -- przekatna:
[[ True False False False]
 [False  True False False]
 [False False  True False]
 [False False False  True]]

~identity_mask -- wszystko OPROCZ przekatnej:
[[False  True  True  True]
 [ True False  True  True]
 [ True  True False  True]
 [ True  True  True False]]

Srednia korelacja (z przekatna, ZLE):    0.6437
Srednia korelacja (bez przekatnej, OK): 0.5250


---
## Podsumowanie -- sciagawka

In [35]:
summary = '''
+===================================================================+
|              NUMPY TRIKI -- SCIAGAWKA                            |
+===================================================================+
| SLICING                                                          |
|   arr[::-1]              odwrocenie (widok, nie kopia)          |
|   arr[::k]                kazdy k-ty element                    |
|   argsort(x)[::-1]        indeksy sortujace MALEJACO             |
+===================================================================+
| NEWAXIS / BROADCASTING                                           |
|   v[:, np.newaxis]        wektor (n,) -> kolumna (n,1)          |
|   v[np.newaxis, :]        wektor (n,) -> wiersz (1,n)           |
|   reguly: porownuj shapes OD PRAWEJ, 1 sie rozciaga              |
+===================================================================+
| GENEROWANIE PROBEK                                                |
|   np.random.normal(loc, scale, size)     loc/scale moga byc tablicami |
|   np.random.standard_normal(size)        szybsze, zawsze N(0,1)  |
|   np.random.multivariate_normal(mu, Sigma, n)  skorelowane probki|
+===================================================================+
| MNOZENIE                                                          |
|   *      element-wise (z broadcastingiem)                        |
|   @      mnozenie macierzowe / dot product                       |
|   1D @ 1D -> skalar      2D @ 1D -> wektor    2D @ 2D -> macierz|
+===================================================================+
| INDEKSOWANIE                                                      |
|   arr[idx]            permutacja/wybor elementow wg tablicy idx  |
|   matrix[idx]          permutacja WIERSZY                        |
|   matrix[:, idx]       permutacja KOLUMN                         |
|   np.argsort(x)        INDEKSY sortujace (nie wartosci!)         |
+===================================================================+
| BUDOWANIE MACIERZY                                                |
|   np.outer(a,b)        outer[i,j] = a[i]*b[j]                    |
|   np.diag(wektor)      -> macierz diagonalna                     |
|   np.diag(macierz)     -> wektor przekatnej                      |
+===================================================================+
| AXIS (redukcje)                                                   |
|   axis=0   redukuje WIERSZE (zostaja kolumny)                    |
|   axis=1   redukuje KOLUMNY (zostaja wiersze)                    |
|   keepdims=True   zachowuje wymiar dla broadcastingu po redukcji|
+===================================================================+
| MASKI LOGICZNE                                                    |
|   arr[arr < 0]         filtrowanie wg warunku                    |
|   mask.sum()           liczba elementow spelniajacych warunek    |
|   mask.mean()          proporcja (bo True=1, False=0)            |
|   np.where(cond, a, b) warunkowe przypisanie                     |
|   ~mask                 negacja maski boolowskiej                |
+===================================================================+
'''
print(summary)



+===================================================================+
|              NUMPY TRIKI -- SCIAGAWKA                            |
+===================================================================+
| SLICING                                                          |
|   arr[::-1]              odwrocenie (widok, nie kopia)          |
|   arr[::k]                kazdy k-ty element                    |
|   argsort(x)[::-1]        indeksy sortujace MALEJACO             |
+===================================================================+
| NEWAXIS / BROADCASTING                                           |
|   v[:, np.newaxis]        wektor (n,) -> kolumna (n,1)          |
|   v[np.newaxis, :]        wektor (n,) -> wiersz (1,n)           |
|   reguly: porownuj shapes OD PRAWEJ, 1 sie rozciaga              |
+===================================================================+
| GENEROWANIE PROBEK                                                |
|   np.random.normal(loc, scale,